# HDB Resale Flat Prices ETL Pipeline (Jan 2012 - Dec 2017)

ETL pipeline for HDB resale flat transactions from data.gov.sg. It produces five output
datasets: **Raw, Cleaned, Transformed, Quarantined** and **Hashed**.

The pipeline code is in `src/` and imported below; this notebook runs it step by step and
shows the results. Setup instructions are in `README.md`, and the findings, quarantine
analysis and assumptions are written up in `reports/data_profiling_report.md`.

**Scope note**: no single data.gov.sg dataset covers Jan 2012 - Dec 2017 exactly. The
pipeline discovers collection 189's datasets, uses every one whose published coverage overlaps
the scope, and keeps rows by their own `month`. No source file is hardcoded (see Section 1).

**Two distinct "strict authoritative set" uses of the Jan 2012 data** (see assumptions in the
report for full rationale):
- *Schema* authority (Req 1): the master dataset's columns are exactly Jan 2012's columns.
- *Domain value* authority (Req 3): Town / Flat Type / Flat Model / storey_range values seen
  in Jan 2012 are the only values any row (any year) is allowed to have; everything else is
  quarantined. This is intentionally strict, and it has one large, visible consequence:
  HDB published **Mar-May 2012 in 5-storey bands** (`01 TO 05`, ...), which the Jan-2012
  domain rejects, so all 6,838 transactions in those three months are absent from the cleaned data.
  Section 4b measures this and the report explains it - read it before building time series.


In [1]:
# Setup: make `src/` importable whether Jupyter was started from the repo root or notebooks/.
import warnings
# Silence one known environment message only (urllib3 on macOS's LibreSSL Python). A blanket
# filter would also hide genuine pandas/pipeline warnings, which should stay visible.
warnings.filterwarnings("ignore", message="urllib3 v2 only supports OpenSSL")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display, Markdown

# The only configuration is what the brief itself specifies: collection 189, the Jan 2012 -
# Dec 2017 scope, the authoritative month and the lease assumptions. No source file is named.
from src.schema import (
    COLLECTION_ID, AUTHORITATIVE_MONTH, SCOPE_START, SCOPE_END,
    REMAINING_LEASE_REFERENCE_DATE, ASSUMED_LEASE_YEARS,
)
from src.ingest import resolve_sources
from src.combine import combine_sources
from src.profiling import profile_dataframe, profile_to_markdown
from src.validation import build_authoritative_domains, validate, split_validated
from src.lease import add_remaining_lease
from src.lease_check import compare_with_published
from src.anomaly import add_anomaly_flags
from src.completeness import monthly_completeness, flagged_months
from src.identifier import build_resale_identifier
from src.deidentify import add_hashed_identifier, ensure_secret_key

# One folder per mandatory output group (Req 11), plus the report.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
TRANSFORMED_DIR = PROJECT_ROOT / "data" / "transformed"
QUARANTINED_DIR = PROJECT_ROOT / "data" / "quarantined"
HASHED_DIR = PROJECT_ROOT / "data" / "hashed"
REPORTS_DIR = PROJECT_ROOT / "reports"

# The hashing step needs a secret key in .env. If there isn't one, or it's still the
# .env.example placeholder, create a random local key so the notebook runs without extra setup.
if ensure_secret_key(PROJECT_ROOT / ".env"):
    print("No usable HASH_SECRET_KEY found - generated a random local key in .env.")
    print("Hashed identifiers will differ from the committed outputs; that is expected.")

pd.set_option("display.max_columns", 50)
print(f"Project root: {PROJECT_ROOT}")
print(f"Collection: {COLLECTION_ID} | Scope: {SCOPE_START} to {SCOPE_END} | Authoritative month: {AUTHORITATIVE_MONTH}")

Project root: /Users/mintsheen.chua/Documents/resale-flat-prices-pipeline
Collection: 189 | Scope: 2012-01 to 2017-12 | Authoritative month: 2012-01


## 1. Dataset Extraction

Nothing about the source files is hardcoded. The pipeline lists collection 189's datasets
through the data.gov.sg metadata API and keeps those whose published coverage overlaps
Jan 2012 - Dec 2017, so the 1990-1999 file is never downloaded. Each kept file is **streamed**
to `data/raw/` exactly as published: no row, column or value is changed.

`data/raw/manifest.json` records each file's SHA-256. Re-runs reuse the committed files
instead of re-downloading. That matters because the 2017-onwards dataset is republished daily,
so a fresh download is a different file. A re-run also stops if any file no longer matches
its checksum. If data.gov.sg is unreachable, the run falls back to the manifest, so the
notebook works offline.

In [2]:
# Discover -> select by coverage -> download only what isn't cached -> verify checksums.
# Once data/raw/manifest.json exists, this cell does no downloading.
sources = resolve_sources(COLLECTION_ID, SCOPE_START, SCOPE_END, RAW_DIR)

pd.DataFrame([{
    "dataset": s.name,
    "published coverage": f"{s.coverage_start} to {s.coverage_end}",
    "file": s.filename,
    "MB": round(s.size_bytes / 1e6, 1),
    "sha256": s.sha256[:12] + "...",
} for s in sources])

,dataset,published coverage,file,MB,sha256
0,"Resale Flat Prices (Based on Approval Date), 2...",2000-01 to 2012-02,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,29.4,5f6a72bd7b91...
1,Resale Flat Prices (Based on Registration Date...,2012-03 to 2014-12,ResaleFlatPricesBasedonRegistrationDateFromMar...,4.2,6a16e2dc78f7...
2,Resale Flat Prices (Based on Registration Date...,2015-01 to 2016-12,ResaleFlatPricesBasedonRegistrationDateFromJan...,3.1,4ff7ce4a4f64...
3,Resale flat prices based on registration date ...,2017-01 to 2026-10,ResaleFlatPricesBasedonRegistrationDateFromJan...,23.9,945e09d75efb...


## 2. Combine into a Single Master Dataset (Req 1)

Rows are kept when their own `month` value falls in scope, so no per-file date ranges are
needed. The master schema is the column set of whichever file contains Jan 2012, the
authoritative set. Later files carry an extra `remaining_lease` column HDB began publishing
afterwards. It isn't carried forward, because the pipeline derives its own in Section 5.

Two guards replace what hardcoded per-file ranges used to guarantee: no month may come from
two files (that would double-count it), and exactly one file must contain Jan 2012.

In [3]:
# combine_sources filters rows by month, enforces both guards, and returns per-file raw-stage
# statistics, which feed the lifecycle trace in Section 11 and the report.
combined, source_stats, authoritative_source = combine_sources(
    sources, RAW_DIR,
    scope_start=SCOPE_START, scope_end=SCOPE_END, authoritative_month=AUTHORITATIVE_MONTH,
)
print(f"Master schema taken from: {authoritative_source.name}")
print(f"Combined master dataset: {combined.shape[0]:,} rows x {combined.shape[1]} columns")
display(source_stats)  # raw stage: what each file held, and how much of it is in scope
combined.head()

Master schema taken from: Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012
Combined master dataset: 113,053 rows x 10 columns


,dataset,raw_rows,raw_months,in_scope_rows,out_of_scope_rows,raw_columns,columns_not_carried
0,"Resale Flat Prices (Based on Approval Date), 2...",369651,2000-01 to 2012-02,3188,366463,10,-
1,Resale Flat Prices (Based on Registration Date...,52203,2012-03 to 2014-12,52203,0,10,-
2,Resale Flat Prices (Based on Registration Date...,37153,2015-01 to 2016-12,37153,0,11,remaining_lease
3,Resale flat prices based on registration date ...,242032,2017-01 to 2026-10,20509,221523,11,remaining_lease


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
0,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,257800.0
1,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,263000.0
2,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,275000.0
3,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,260000.0
4,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,226000.0


## 3. Data Profiling (Req 2)

A lightweight custom profiler (`src/profiling.py`) - dtypes, missingness, cardinality, numeric
stats, and top categorical values - rather than a heavyweight framework, so the report stays
auditable. The full profiling output is also written to `reports/data_profiling_report.md`.


In [4]:
# Profile the combined data before any rule is applied. This is the baseline that the cleaned
# and quarantined profiles are compared against in the report.
combined_profile = profile_dataframe(combined)
display(Markdown(profile_to_markdown(combined_profile, "Combined master dataset profile")))

### Combined master dataset profile

- Rows: **113,053** | Columns: **10**
- Month range: **2012-01 to 2017-12**
- Exact duplicate rows (both/all copies counted): **902**

| Column | dtype | % missing | # unique | notes |
|---|---|---|---|---|
| month | object | 0.0% | 72 |  |
| town | object | 0.0% | 26 | JURONG WEST=9193, WOODLANDS=8930, TAMPINES=8027 |
| flat_type | object | 0.0% | 7 | 4 ROOM=45139, 3 ROOM=31354, 5 ROOM=26370 |
| block | object | 0.0% | 2193 |  |
| street_name | object | 0.0% | 526 |  |
| storey_range | object | 0.0% | 25 | 04 TO 06=25934, 07 TO 09=23193, 01 TO 03=21036 |
| floor_area_sqm | float64 | 0.0% | 169 | min=31.0, max=280.0, mean=96.8 |
| flat_model | object | 0.0% | 20 | Model A=32702, Improved=29175, New Generation=19497 |
| lease_commence_date | int64 | 0.0% | 50 | min=1966.0, max=2015.0, mean=1990.6 |
| resale_price | float64 | 0.0% | 2865 | min=175000.0, max=1180000.0, mean=449659.9 |


## 4. Data Validation & Quarantine (Req 3, 6)

Authoritative value sets for Town / Flat Type / Flat Model / storey_range are built from the
**Jan 2012 slice only**. Every row in the full combined dataset (all years) is checked against
these sets, plus a handful of additional sanity rules: required fields present, resale_price
> 0, floor_area_sqm within a plausible band, month within scope, and exact full-row duplicates
(first occurrence kept as valid, the rest quarantined). Rows are quarantined with a
`quarantine_reason` tag rather than silently dropped.


In [5]:
# The allowed values come only from the Jan 2012 rows (Req 3). build_authoritative_domains
# raises if this slice is ever empty; otherwise every row would be quietly quarantined.
jan_2012_slice = combined[combined["month"] == AUTHORITATIVE_MONTH]
print(f"Jan 2012 authoritative slice: {len(jan_2012_slice):,} rows")

domains = build_authoritative_domains(jan_2012_slice)
for col, values in domains.items():
    print(f"  {col}: {len(values)} allowed values")

# Each row gets a quarantine_reason listing every rule it failed; an empty reason means it
# passed. Rows are split on that, never dropped.
validated = validate(combined, domains, scope_start=SCOPE_START, scope_end=SCOPE_END)
cleaned, quarantined = split_validated(validated)

print(f"\nCleaned:     {len(cleaned):,} rows")
print(f"Quarantined: {len(quarantined):,} rows")
# Invariant: every combined row lands in exactly one of the two outputs.
assert len(cleaned) + len(quarantined) == len(combined), "row reconciliation failed"
print("Row reconciliation OK: cleaned + quarantined == combined")

Jan 2012 authoritative slice: 1,559 rows
  town: 26 allowed values
  flat_type: 7 allowed values
  flat_model: 13 allowed values
  storey_range: 12 allowed values

Cleaned:     104,783 rows
Quarantined: 8,270 rows
Row reconciliation OK: cleaned + quarantined == combined


In [6]:
# Reasons are listed per row, so a row failing two rules is counted under the joint reason.
print("Quarantine reasons breakdown:")
quarantined["quarantine_reason"].value_counts()

Quarantine reasons breakdown:


quarantine_reason
invalid_storey_range                       6952
invalid_flat_model                          775
duplicate_record                            442
invalid_flat_model;invalid_storey_range      92
invalid_storey_range;duplicate_record         6
invalid_flat_model;duplicate_record           3
Name: count, dtype: int64

### 4b. Monthly completeness - what the strict rule costs

Row reconciliation (cleaned + quarantined == combined) proves nothing was *lost*, but not that
what survived is usable. A rule that rejects most of one month leaves a hole in the time series
that reconciles perfectly and is invisible in the totals. So retention is checked **per month**,
flagging any month that is missing or keeps less than half its rows (`src/completeness.py`).

The same check is what the operationalised pipeline uses as its **publish gate** in Part 2: a
flagged month should stop publication and alert, not just be reported after the fact.

In [7]:
# Retention per month. The reconciliation above can't see a month that was hollowed out
# by validation; this can, and it flags any month that is missing or keeps under half its rows.
completeness = monthly_completeness(combined, cleaned, scope_start=SCOPE_START, scope_end=SCOPE_END)
gaps = flagged_months(completeness)
print(f"Months flagged: {len(gaps)} of {len(completeness)} in scope")
display(gaps)

# Root cause: which storey_range values did the flagged months use?
gap_quarantine = quarantined[quarantined["month"].isin(gaps["month"])]
print("storey_range values among the flagged months' quarantined rows:")
display(gap_quarantine["storey_range"].value_counts().head(8).to_frame("rows"))
print("Jan-2012 authoritative storey_range domain:", sorted(domains["storey_range"]))

Months flagged: 3 of 72 in scope


,month,source_rows,cleaned_rows,retention_pct,flag
0,2012-03,2360,0,0.0,retention_below_50pct
1,2012-04,2155,0,0.0,retention_below_50pct
2,2012-05,2323,0,0.0,retention_below_50pct


storey_range values among the flagged months' quarantined rows:


,rows
storey_range,
01 TO 05,2700
06 TO 10,2474
11 TO 15,1259
16 TO 20,265
21 TO 25,92
26 TO 30,39
36 TO 40,7
31 TO 35,2


Jan-2012 authoritative storey_range domain: ['01 TO 03', '04 TO 06', '07 TO 09', '10 TO 12', '13 TO 15', '16 TO 18', '19 TO 21', '22 TO 24', '25 TO 27', '28 TO 30', '31 TO 33', '34 TO 36']


## 5. Remaining Lease Calculation (Req 4)

Assumption: HDB lease is 99 years from `lease_commence_date` (a year only in the source data,
so lease start is taken as 1 Jan of that year). Remaining lease is computed as of
**1 Jan 2027**, floored down to whole years and whole months.


In [8]:
# The source gives only a lease commencement *year*, so the lease is taken to start on 1 Jan.
# Start and expiry therefore both fall on 1 January, and the month component is always 0.
# That is checked here rather than assumed.
cleaned = add_remaining_lease(cleaned)
print(f"remaining_lease_months unique values: {sorted(cleaned['remaining_lease_months'].unique())}")
print("(always 0 - a correct side effect of lease_commence_date only having year granularity; see src/lease.py)")
cleaned[["lease_commence_date", "remaining_lease_years", "remaining_lease_months", "remaining_lease_display"]].head()

remaining_lease_months unique values: [np.int64(0)]
(always 0 - a correct side effect of lease_commence_date only having year granularity; see src/lease.py)


,lease_commence_date,remaining_lease_years,remaining_lease_months,remaining_lease_display
0,1979,51,0,51 years 0 months
1,1978,50,0,50 years 0 months
2,1978,50,0,50 years 0 months
3,1986,58,0,58 years 0 months
4,1986,58,0,58 years 0 months


### 5b. Validating the 99-year assumption against HDB's own figures

The brief tells us to *assume* a 99-year lease. That assumption is testable: the 2015-onwards
source files publish HDB's own `remaining_lease`. It is deliberately not part of the master
dataset (the Jan-2012 schema is authoritative and predates that column), but it can be read
straight from the raw files as an independent reference.

The useful output is not just "is 99 years right?" but **how precise our derived figure can
possibly be** - since `lease_commence_date` is a *year*, our value can be off by up to 12
months against the true lease start date. That is an error bar the Data Science Team should
know about, so it is measured rather than asserted.


In [9]:
# HDB's own remaining_lease (2015-onwards files only) is read from the raw files as an
# independent check. It is never added to the master dataset.
lease_check = compare_with_published(RAW_DIR, sources, scope_start=SCOPE_START, scope_end=SCOPE_END)
display(lease_check)
print("Interpretation: agreement is effectively total at whole-year resolution, which confirms the")
print("99-year assumption. The residual sub-year spread is the precision limit of a year-only")
print("lease_commence_date - not a calculation error. Treat remaining_lease_years as +/- 1 year.")

,source,rows_compared,published_format,exact_match_pct,within_12_months_pct,median_abs_diff_months,max_abs_diff_months
0,Resale Flat Prices (Based on Registration Date...,37153,70,6.8,100.0,4,13
1,Resale flat prices based on registration date ...,20509,61 years 04 months,11.1,99.9,6,14


Interpretation: agreement is effectively total at whole-year resolution, which confirms the
99-year assumption. The residual sub-year spread is the precision limit of a year-only
lease_commence_date - not a calculation error. Treat remaining_lease_years as +/- 1 year.


## 6. Anomalous Resale Price Detection (Req 5)

Tukey's IQR fences (1.5x IQR beyond Q1/Q3), computed **per (town, flat_type, flat_model)
group** rather than globally (a global threshold would just rediscover which towns are
expensive). Checked on both raw `resale_price` and size-normalised `price_per_sqm`. This flags
rows for review - it does not quarantine them, since "potentially anomalous" is not the same
as invalid.


In [10]:
# Outliers are judged within (town, flat_type, flat_model) groups, on price and on
# price-per-sqm. Flagged rows stay in the cleaned data: "potentially anomalous" is not invalid.
cleaned = add_anomaly_flags(cleaned)
n_anomalous = cleaned["is_price_anomalous"].sum()
print(f"Flagged as potentially anomalous: {n_anomalous:,} of {len(cleaned):,} rows ({100*n_anomalous/len(cleaned):.2f}%)")
cleaned[cleaned["is_price_anomalous"]][["month", "town", "flat_type", "flat_model", "resale_price", "price_per_sqm", "price_anomaly_reason"]].head(10)

Flagged as potentially anomalous: 1,047 of 104,783 rows (1.00%)


,month,town,flat_type,flat_model,resale_price,price_per_sqm,price_anomaly_reason
266,2012-01,BUKIT MERAH,2 ROOM,Standard,442000.0,6597.014925,resale_price_outlier_in_group
392,2012-01,BUKIT TIMAH,5 ROOM,Improved,658000.0,5393.442623,resale_price_outlier_in_group;price_per_sqm_ou...
534,2012-01,GEYLANG,3 ROOM,Standard,373000.0,6016.129032,resale_price_outlier_in_group
554,2012-01,GEYLANG,EXECUTIVE,Apartment,682000.0,4736.111111,price_per_sqm_outlier_in_group
629,2012-01,HOUGANG,EXECUTIVE,Apartment,475000.0,3125.000000,price_per_sqm_outlier_in_group
904,2012-01,PASIR RIS,5 ROOM,Model A,648000.0,4800.000000,resale_price_outlier_in_group;price_per_sqm_ou...
918,2012-01,PASIR RIS,EXECUTIVE,Maisonette,840000.0,4468.085106,resale_price_outlier_in_group
1298,2012-01,TOA PAYOH,5 ROOM,Model A-Maisonette,725000.0,4865.771812,resale_price_outlier_in_group
1402,2012-01,WOODLANDS,5 ROOM,Improved,535000.0,4314.516129,resale_price_outlier_in_group
1490,2012-01,YISHUN,3 ROOM,New Generation,393000.0,5865.671642,resale_price_outlier_in_group;price_per_sqm_ou...


## 7. Save Cleaned & Quarantined Outputs


In [11]:
# Cleaned: rows that passed every rule, plus the derived lease and anomaly columns.
# Quarantined: failed rows with their reasons, so each decision can be reviewed or reversed.
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
QUARANTINED_DIR.mkdir(parents=True, exist_ok=True)

cleaned_path = CLEANED_DIR / "cleaned_resale_flat_prices.csv"
quarantined_path = QUARANTINED_DIR / "quarantined_resale_flat_prices.csv"

cleaned.to_csv(cleaned_path, index=False)
quarantined.to_csv(quarantined_path, index=False)
print(f"Wrote {cleaned_path} ({len(cleaned):,} rows)")
print(f"Wrote {quarantined_path} ({len(quarantined):,} rows)")

Wrote /Users/mintsheen.chua/Documents/resale-flat-prices-pipeline/data/cleaned/cleaned_resale_flat_prices.csv (104,783 rows)
Wrote /Users/mintsheen.chua/Documents/resale-flat-prices-pipeline/data/quarantined/quarantined_resale_flat_prices.csv (8,270 rows)


## 8. Transformation - Resale Identifier (Req 8)

A 12-character code per row: `S` + 3-digit block prefix + 3-digit (month, town, flat_type)
average-price prefix + `MMYY` month code + town initial. Collisions are expected (many
transactions can share all four components) and are resolved deterministically by suffixing
repeats (`_01`, `_02`, ...) in original row order - **no rows are dropped**.


In [12]:
# Built from the validated data, as Req 8 specifies. The price digits use the mean resale
# price of each (month, town, flat_type) group, not the row's own price.
transformed = build_resale_identifier(cleaned)
print(f"Unique identifiers: {transformed['resale_identifier'].is_unique} ({transformed['resale_identifier'].nunique():,} / {len(transformed):,})")

# One collision, worked through. Rows sharing every identifier component get _01, _02, ...
# suffixes in original row order: unique without dropping any record.
dupe_base = transformed["resale_identifier"].str.slice(0, 12)
example_base = dupe_base[dupe_base.duplicated(keep=False)].iloc[0]
transformed[dupe_base == example_base][["block", "month", "town", "flat_type", "resale_price", "resale_identifier"]]

Unique identifiers: True (104,783 / 104,783)


,block,month,town,flat_type,resale_price,resale_identifier
1,314,2012-01,ANG MO KIO,2 ROOM,263000.0,S3142561201A
2,314,2012-01,ANG MO KIO,2 ROOM,275000.0,S3142561201A_01


## 9. De-identification - Hashing the Identifier (Req 9)

`HMAC-SHA256(key=secret, msg=resale_identifier)` - irreversible, preserves uniqueness, and
(unlike a plain or publicly-salted hash) resists dictionary attacks: the identifier's
construction rule is simple and its component space is small/enumerable, so without a secret
key an attacker could otherwise precompute every possible identifier -> hash pair. Full
rationale in `src/deidentify.py` and the profiling report.


In [13]:
# HMAC-SHA256 keyed with the secret in .env: one-way, unique per identifier, and not
# reproducible by enumeration without the key, even though the identifier's rule is public.
hashed = add_hashed_identifier(transformed)
print(f"Unique hashed identifiers: {hashed['hashed_identifier'].is_unique} ({hashed['hashed_identifier'].nunique():,} / {len(hashed):,})")
hashed[["resale_identifier", "hashed_identifier"]].head()

Unique hashed identifiers: True (104,783 / 104,783)


,resale_identifier,hashed_identifier
0,S4062561201A,3a1d2488f9de3ef4906bfe73c7c44ab49fa008b78eedc2...
1,S3142561201A,8d8f22a9a5d8df8ae4547801c45096629a5b113d78f8e5...
2,S3142561201A_01,d14ab724dc50a17d2fdb52b035ba11c54de03c1f6764bb...
3,S1702561201A,38d70a5832ed08f101e06cb84aa37a57420767977f5004...
4,S1742561201A,389bfd683fadcc4330b2f5f925c6c5cd64bc1a144a1288...


## 10. Save Transformed & Hashed Outputs

Transformed keeps the plaintext `resale_identifier`. Hashed carries `hashed_identifier`
**only** (plaintext identifier dropped) - otherwise de-identification would be moot.


In [14]:
# Transformed keeps the plaintext identifier. Hashed drops it, because shipping both in the
# same file would undo the de-identification.
TRANSFORMED_DIR.mkdir(parents=True, exist_ok=True)
HASHED_DIR.mkdir(parents=True, exist_ok=True)

transformed_path = TRANSFORMED_DIR / "transformed_resale_flat_prices.csv"
hashed_path = HASHED_DIR / "hashed_resale_flat_prices.csv"

transformed.to_csv(transformed_path, index=False)
hashed_only = hashed.drop(columns=["resale_identifier"])
hashed_only.to_csv(hashed_path, index=False)

print(f"Wrote {transformed_path} ({len(transformed):,} rows)")
print(f"Wrote {hashed_path} ({len(hashed_only):,} rows, columns: {list(hashed_only.columns)})")

Wrote /Users/mintsheen.chua/Documents/resale-flat-prices-pipeline/data/transformed/transformed_resale_flat_prices.csv (104,783 rows)
Wrote /Users/mintsheen.chua/Documents/resale-flat-prices-pipeline/data/hashed/hashed_resale_flat_prices.csv (104,783 rows, columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease_years', 'remaining_lease_months', 'remaining_lease_display', 'price_per_sqm', 'is_price_anomalous', 'price_anomaly_reason', 'hashed_identifier'])


## 11. Pipeline Lifecycle & Reconciliation

Every row is traced from the files as published to each output group, with the count at
every stage. The assertions make the notebook fail if any stage loses or invents rows. The
same table opens the profiling report, which covers the data's behaviour across the whole
pipeline (Req 14).

In [15]:
# Stage-by-stage row counts. Negative rows are removals, so the table reads as a funnel.
raw_rows = int(source_stats["raw_rows"].sum())
out_of_scope = int(source_stats["out_of_scope_rows"].sum())
n_suffixed = int(transformed["resale_identifier"].str.contains("_").sum())

lifecycle = pd.DataFrame([
    {"stage": "Raw (as published)", "rows": raw_rows,
     "detail": f"{len(sources)} files discovered in collection {COLLECTION_ID}; stored unmodified"},
    {"stage": "  out of scope", "rows": -out_of_scope,
     "detail": f"months outside {SCOPE_START} to {SCOPE_END}; removed in memory only"},
    {"stage": "Combined", "rows": len(combined),
     "detail": "master dataset on the Jan 2012 schema"},
    {"stage": "  quarantined", "rows": -len(quarantined),
     "detail": "failed validation or duplicate; kept with reasons in quarantined/"},
    {"stage": "Cleaned", "rows": len(cleaned),
     "detail": "+ remaining lease, price_per_sqm, anomaly flags"},
    {"stage": "Transformed", "rows": len(transformed),
     "detail": f"+ resale_identifier ({n_suffixed:,} rows needed a collision suffix)"},
    {"stage": "Hashed", "rows": len(hashed_only),
     "detail": "resale_identifier replaced by hashed_identifier"},
])

# Each stage must reconcile with the one before it.
assert raw_rows - out_of_scope == len(combined), "scope filter lost or invented rows"
assert len(cleaned) + len(quarantined) == len(combined), "validation lost rows"
assert len(cleaned) == len(transformed) == len(hashed_only), "transformation changed the row count"
assert transformed["resale_identifier"].is_unique and hashed_only["hashed_identifier"].is_unique
assert "resale_identifier" not in hashed_only.columns, "plaintext identifier leaked into hashed output"
print("All reconciliation checks passed.")
lifecycle

All reconciliation checks passed.


,stage,rows,detail
0,Raw (as published),701039,4 files discovered in collection 189; stored u...
1,out of scope,-587986,months outside 2012-01 to 2017-12; removed in ...
2,Combined,113053,master dataset on the Jan 2012 schema
3,quarantined,-8270,failed validation or duplicate; kept with reas...
4,Cleaned,104783,"+ remaining lease, price_per_sqm, anomaly flags"
5,Transformed,104783,"+ resale_identifier (12,952 rows needed a coll..."
6,Hashed,104783,resale_identifier replaced by hashed_identifier


## 12. Generate the Data Profiling Report (Req 7, 14)

Writes `reports/data_profiling_report.md`: dataset overview, profiling of the combined/
cleaned/quarantined datasets, the quarantine-reason breakdown, anomaly findings, the
identifier/hashing design rationale, and a dedicated Assumptions section. This single file
covers both the "insights & assumptions" requirement and the "profiling report across the
pipeline lifecycle + quarantine bucket" requirement, since their content overlaps heavily -
documented here as a deliberate choice rather than duplicating across two files.


In [16]:
# The report is generated from this run's data, so its numbers can't drift from the outputs.
cleaned_profile = profile_dataframe(cleaned)
quarantined_profile = profile_dataframe(quarantined)

report_sections = []
report_sections.append("# HDB Resale Flat Prices - Data Profiling & Insights Report\n")
report_sections.append(
    f"Pipeline scope: **{SCOPE_START} to {SCOPE_END}**. "
    f"Authoritative month for schema and domain validation: **{AUTHORITATIVE_MONTH}**.\n"
)

report_sections.append("## Dataset Overview\n")
report_sections.append(
    f"- Combined master dataset (pre-validation): **{len(combined):,}** rows\n"
    f"- Cleaned: **{len(cleaned):,}** rows\n"
    f"- Quarantined: **{len(quarantined):,}** rows "
    f"({100*len(quarantined)/len(combined):.2f}% of combined)\n"
    f"- Transformed / Hashed: **{len(transformed):,}** rows each (all Cleaned rows carry a "
    "unique identifier)\n"
)
if len(gaps):
    report_sections.append(
        f"\n> **Known gap - read before using the cleaned data:** {len(gaps)} in-scope month(s) "
        f"retain under half their rows after validation: "
        + ", ".join(f"**{r.month}** ({r.retention_pct:g}% retained)" for r in gaps.itertuples())
        + ". Any time series built from `cleaned/` will show a hole there. It is an upstream "
        "reporting-convention change, not a fall in transactions - see *Quarantine Bucket "
        "Analysis* below.\n"
    )

# Lifecycle first: a reader should see where every row went before the per-stage profiles.
report_sections.append("## Pipeline Lifecycle\n")
report_sections.append(
    "Every row, from the files as published to each output group. The notebook asserts that "
    "each stage reconciles with the one before it.\n\n"
    + lifecycle.assign(rows=lifecycle["rows"].map(lambda n: f"{n:,}")).to_markdown(index=False)
    + "\n"
)

report_sections.append("### Stage 1 - Raw: the source files\n")
report_sections.append(
    f"Discovered from collection {COLLECTION_ID} by each dataset's published coverage; stored "
    "byte-for-byte and verified against the checksums in `data/raw/manifest.json` on every run.\n\n"
    + source_stats.to_markdown(index=False) + "\n\n"
    f"- {out_of_scope:,} of {raw_rows:,} published rows ({out_of_scope / raw_rows:.0%}) are out of scope: "
    "the 2000-2011 history in the first file, and post-2017 rows in the open-ended last file. "
    "They are removed in memory; the files themselves are untouched.\n"
    "- Files disagree on schema: two carry a `remaining_lease` column the Jan 2012 file lacks. "
    "It is excluded from the master dataset, but used as an independent reference in the lease "
    "validation below - where it also turns out to be formatted differently between files.\n"
    "- Upstream naming isn't stable either: data.gov.sg has changed the casing of one published "
    "filename, which the ingestion step handles so the manifest stays valid on case-sensitive "
    "filesystems.\n"
)

report_sections.append("### Stages 2-4 - Combined, Cleaned, Quarantined\n")
report_sections.append(profile_to_markdown(combined_profile, "Combined master dataset profile"))
report_sections.append(profile_to_markdown(cleaned_profile, "Cleaned dataset profile"))
report_sections.append(profile_to_markdown(quarantined_profile, "Quarantined dataset profile"))

# Transformed and Hashed add one column each, so profile what those columns contain.
report_sections.append("### Stages 5-6 - Transformed and Hashed\n")
base_counts = transformed["resale_identifier"].str.slice(0, 12).value_counts()
report_sections.append(
    f"- **Transformed**: {len(transformed):,} rows, one new column `resale_identifier`. "
    f"{base_counts.size:,} distinct base identifiers; {n_suffixed:,} rows "
    f"({n_suffixed / len(transformed):.1%}) needed a collision suffix. The most shared base "
    f"identifier covers {base_counts.max()} rows, so suffixes never exceed two digits.\n"
    f"- **Hashed**: {len(hashed_only):,} rows. `resale_identifier` is removed and replaced by a "
    "64-character `hashed_identifier`, unique across every row (asserted in the notebook).\n"
)

report_sections.append("## Quarantine Bucket Analysis\n")
reason_counts = quarantined["quarantine_reason"].value_counts()
report_sections.append("| Quarantine reason combination | Row count |\n|---|---|\n" + "\n".join(
    f"| {reason} | {count:,} |" for reason, count in reason_counts.items()
) + "\n")
storey_fail = quarantined["quarantine_reason"].str.contains("invalid_storey_range")
model_fail = quarantined["quarantine_reason"].str.contains("invalid_flat_model")
in_gap = quarantined["month"].isin(gaps["month"])
storey_outside = quarantined.loc[storey_fail & ~in_gap, "storey_range"].value_counts()
models = quarantined.loc[model_fail, "flat_model"].value_counts()
source_by_year = combined.groupby(combined["month"].str[:4]).size()
quar_by_year = quarantined.groupby(quarantined["month"].str[:4]).size().reindex(source_by_year.index, fill_value=0)
yearly_rate = (100 * quar_by_year / source_by_year).round(1)

report_sections.append(
    f"\n**Key finding - most of the quarantine bucket is one upstream reporting change, not dirty "
    f"data.** {int((storey_fail & in_gap).sum()):,} of {len(quarantined):,} quarantined rows "
    f"({100 * (storey_fail & in_gap).sum() / len(quarantined):.0f}%) fail `invalid_storey_range` "
    f"inside just {', '.join(gaps['month'])}. In those months HDB published storey ranges in "
    "**5-storey bands** (`01 TO 05`, `06 TO 10`, ...) instead of the 3-storey bands it used in "
    "Jan 2012 and again from mid-2012 onward.\n\n"
    "The strict rule is kept as the spec requires, and deliberately not 'repaired': a 5-storey "
    "band cannot be mapped losslessly onto 3-storey bands (`01 TO 05` straddles `01 TO 03` and "
    "`04 TO 06`), so any mapping would invent precision the source never had.\n\n"
    f"The remaining failures are genuine values absent from Jan 2012:\n"
    f"- storey_range outside those months ({int(storey_outside.sum()):,} rows): "
    + ", ".join(f"`{v}` ({n})" for v, n in storey_outside.head(6).items())
    + " - floors above Jan 2012's highest band (`34 TO 36`).\n"
    f"- flat_model ({int(model_fail.sum()):,} rows): "
    + ", ".join(f"`{v}` ({n})" for v, n in models.items()) + ".\n\n"
    "Quarantine rate by year (% of that year's combined rows):\n\n"
    "| Year | " + " | ".join(yearly_rate.index) + " |\n|---|" + "---|" * len(yearly_rate) + "\n"
    "| Quarantined % | " + " | ".join(f"{v:g}" for v in yearly_rate.values) + " |\n\n"
    "**Consequence for consumers:** those months are missing from the cleaned dataset entirely (see "
    "*Known gap* above). The rows are preserved in `quarantined/` with their reason, so a "
    "consumer who accepts the coarser 5-storey band can reinstate them deliberately. The "
    "per-month completeness check (`src/completeness.py`) makes this visible automatically, "
    f"rather than relying on someone noticing the month count drop from "
    f"{combined['month'].nunique()} to {cleaned['month'].nunique()}.\n"
)

report_sections.append("## Are the quarantined duplicates errors, or real repeat transactions?\n")
n_dups = int(quarantined["quarantine_reason"].str.contains("duplicate_record").sum())
report_sections.append(
    f"{n_dups:,} rows were quarantined as `duplicate_record`. This is a judgment call worth "
    "stating plainly: the dataset carries **no transaction identifier**, and its granularity is "
    "coarse (month, not date; a 3-storey band, not a unit number). Two genuinely different flats "
    "in the same block and storey band, of the same size, sold in the same month at the same "
    "price would be indistinguishable from one record written twice.\n\n"
    "The pipeline keeps the first occurrence and quarantines the rest, so the cleaned dataset "
    "never double-counts. Because quarantine preserves the rows rather than discarding them, the "
    "Data Science Team can reinstate them if they judge repeats to be genuine - the decision is "
    "reversible, which is why quarantining was preferred to dropping.\n"
)

report_sections.append("## Validating the 99-year lease assumption\n")
report_sections.append(
    "The brief states the 99-year lease as an assumption; the 2015-onwards source files let us "
    "test it against HDB's own published `remaining_lease` (a column deliberately excluded from "
    "the master dataset, since the authoritative Jan-2012 schema predates it).\n\n"
    + lease_check.to_markdown(index=False) + "\n\n"
    "**Finding**: agreement is effectively total at whole-year resolution, so the 99-year "
    "assumption holds. The residual sub-year spread is not an error - it is the precision limit "
    "of a `lease_commence_date` given as a *year*, so the true lease start can sit anywhere in "
    "that year. **`remaining_lease_years` should be read as +/- 1 year**, and that is also why "
    "`remaining_lease_months` is always 0.\n\n"
    "A secondary observation: the published column is not consistently formatted across vintages "
    "- whole years as a bare integer in the 2015-2016 file, free text (`61 years 04 months`) from "
    "2017 onward. That inconsistency is a further argument for deriving the value ourselves "
    "rather than depending on the source column.\n"
)

report_sections.append("## Anomalous Resale Price Findings\n")
report_sections.append(
    f"- {int(cleaned['is_price_anomalous'].sum()):,} of {len(cleaned):,} cleaned rows "
    f"({100*cleaned['is_price_anomalous'].mean():.2f}%) flagged as potentially anomalous via "
    "Tukey's IQR fences (1.5x IQR), computed per (town, flat_type, flat_model) group on both "
    "resale_price and price_per_sqm. Flagged, not quarantined - see `src/anomaly.py` for the "
    "full rationale.\n"
)

report_sections.append("## Resale Identifier & De-identification\n")
n_base_collisions = int(transformed['resale_identifier'].str.contains('_').sum())
report_sections.append(
    f"- {n_base_collisions:,} rows required a collision-resolution suffix on the base 12-character "
    "identifier (see `src/identifier.py`). Suffixes are deterministic and assigned in original "
    "row order - no rows were dropped to achieve uniqueness.\n"
    "- The identifier is de-identified via HMAC-SHA256 keyed with a secret from `.env` "
    "(`HASH_SECRET_KEY`), not a plain or publicly-salted hash, specifically because the "
    "identifier's construction rule is simple/enumerable - full rationale in "
    "`src/deidentify.py`.\n"
)

report_sections.append("## Data dictionary — columns added by this pipeline\n")
report_sections.append(
    "The ten source columns are carried through unchanged. The columns below are derived, and "
    "are what the Data Science Team is actually being handed.\n\n"
    "| Column | Type | Present in | Meaning |\n|---|---|---|---|\n"
    "| `remaining_lease_years` | int | cleaned, transformed, hashed | Whole years of lease left at 2027-01-01, floored. Read as +/- 1 year (see lease validation above). |\n"
    "| `remaining_lease_months` | int | cleaned, transformed, hashed | Month component; always 0, because `lease_commence_date` is a year only. |\n"
    "| `remaining_lease_display` | str | cleaned, transformed, hashed | Human-readable form, e.g. `51 years 0 months`. |\n"
    "| `price_per_sqm` | float | cleaned, transformed, hashed | `resale_price / floor_area_sqm`. Size-normalised price, used by the anomaly rule and useful for modelling. |\n"
    "| `is_price_anomalous` | bool | cleaned, transformed, hashed | True if the row is a price outlier **within its own (town, flat_type, flat_model) group**. A flag for review, not a quality failure — these rows are valid. |\n"
    "| `price_anomaly_reason` | str | cleaned, transformed, hashed | Which test fired: `resale_price_outlier_in_group`, `price_per_sqm_outlier_in_group`, or both. |\n"
    "| `resale_identifier` | str | transformed only | The 12-character business identifier, with a `_NN` suffix where the base code collided. |\n"
    "| `hashed_identifier` | str | hashed only | HMAC-SHA256 of `resale_identifier`. Replaces the plaintext identifier — de-identification would be pointless if both appeared together. |\n"
    "| `quarantine_reason` | str | quarantined only | Semicolon-joined list of every rule the row failed. |\n\n"
    "**Which output should the Data Science Team use?** `hashed/` for general analysis — it is the "
    "full cleaned dataset with a non-reversible key. Use `transformed/` only if the plaintext "
    "identifier is genuinely needed. `quarantined/` is a data-engineering triage surface, not an "
    "analysis input.\n"
)

report_sections.append("## Assumptions\n")
report_sections.append(
    "1. **Two distinct uses of \"Jan 2012 as strict authoritative set\"**: schema authority "
    "(Req 1, master dataset columns = Jan 2012's columns) is treated separately from domain-"
    "value authority (Req 3, valid Town/Flat Type/Flat Model/storey_range values = those "
    "observed in Jan 2012). See notebook Section 2.\n"
    "2. **Source file selection**: no single published dataset covers Jan 2012-Dec 2017, so "
    "the pipeline discovers collection 189's datasets at run time, uses every one whose "
    "published coverage overlaps the scope, and keeps rows by their own `month`. It fails rather "
    "than combine two files that overlap in time, because that would double-count. Raw files "
    "are pinned by checksum, so re-runs reproduce this exact snapshot rather than silently "
    "picking up data.gov.sg's daily republication of the 2017-onwards file.\n"
    "3. **Remaining lease**: lease start = 1 Jan of `lease_commence_date`'s year; lease term = "
    "99 years; reference date = 1 Jan 2027; result floored to whole years+months (Req 4). "
    "Side effect worth flagging explicitly: because lease start/expiry always fall on 1 "
    "January, `remaining_lease_months` is always 0 for every row - that's correct given "
    "year-only source data, not a calculation bug.\n"
    "4. **Duplicate handling**: exact full-row duplicates keep their first occurrence as a "
    "valid Cleaned row; subsequent copies are quarantined as `duplicate_record`.\n"
    "5. **Anomaly detection is a flag, not a quarantine rule**: \"potentially anomalous\" "
    "(Req 5) is treated as weaker than \"invalid\" (Req 3) - anomalous rows stay in "
    "Cleaned/Transformed/Hashed with a flag for the Data Science Team to decide on.\n"
    "6. **Sanity bounds**: floor_area_sqm accepted in [20, 500] sqm; resale_price must be > 0; "
    "chosen generously from the observed data range so they catch only clearly implausible "
    "values, not legitimate edge cases.\n"
)

report_path = REPORTS_DIR / "data_profiling_report.md"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(report_sections))
print(f"Wrote {report_path} ({report_path.stat().st_size:,} bytes)")


Wrote /Users/mintsheen.chua/Documents/resale-flat-prices-pipeline/reports/data_profiling_report.md (16,808 bytes)
